# Worked Capstone: Operations Anomaly Detection and Alert Triage

**Domain:** Unsupervised/semi-supervised operations analytics  
**Primary dataset:** `operations_anomalies.csv`  
**Level:** Practitioner to Advanced

## Business goal

Rank unusual operating intervals for review while controlling alert volume and retaining interpretable evidence.

This is a worked reference project. First attempt the corresponding phase project independently; then use this capstone to compare framing, evaluation, code structure, and communication.

## Decision questions

        1. What is anomalous relative to context?
2. How should the review budget set a threshold?
3. Which features support each alert?
4. How does ranking perform on available labels?

        ## Definition of done

        - [ ] Data audit
- [ ] Context features
- [ ] Isolation Forest
- [ ] Budget threshold
- [ ] Label evaluation
- [ ] Alert table
- [ ] Monitoring limitations

## End-to-end workflow

```text
Decision and scope
      ↓
Data contract and quality
      ↓
Exploration and hypotheses
      ↓
Baseline and evaluation design
      ↓
Candidate method(s)
      ↓
Held-out / temporal evaluation
      ↓
Error, slice, and sensitivity analysis
      ↓
Artifacts, limitations, recommendation
```

At every stage, distinguish calculation correctness, statistical validity, operational validity, and decision validity.

## Risk register

        | Risk | Mitigation |
        |---|---|
        | Normal seasonality flagged as anomaly | Add time/context features or segment baselines. |
| Rare but harmless events dominate | Validate against impact and reviewer feedback. |
| Fixed contamination drifts | Monitor score distribution and alert yield. |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## 1. Load and inspect time context

Sort time and inspect labels only for evaluation, not unsupervised fitting.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score,average_precision_score,classification_report

ops=pd.read_csv(DATA_DIR/"operations_anomalies.csv",parse_dates=["timestamp"]).sort_values("timestamp")
ops["hour_sin"]=np.sin(2*np.pi*ops.timestamp.dt.hour/24)
ops["hour_cos"]=np.cos(2*np.pi*ops.timestamp.dt.hour/24)
features=["latency_ms","error_rate","throughput_rpm","cpu_pct","memory_pct","hour_sin","hour_cos"]
display(ops[features+["is_anomaly"]].describe().T)

## 2. Fit and score

Robust scaling reduces domination by extreme values; Isolation Forest returns a continuous ranking.

In [ ]:
X=RobustScaler().fit_transform(ops[features])
detector=IsolationForest(n_estimators=300,max_samples="auto",contamination="auto",random_state=42).fit(X)
ops["anomaly_score"]=-detector.score_samples(X)
print("Score quantiles:")
display(ops.anomaly_score.quantile([.5,.9,.95,.97,.99,.995]).to_frame())

## 3. Review-budget threshold

Assume reviewers can inspect the top 3% of intervals. Operational thresholds should be validated by alert value and capacity.

In [ ]:
review_fraction=.03
threshold=ops.anomaly_score.quantile(1-review_fraction)
ops["alert"]=(ops.anomaly_score>=threshold).astype(int)
print("Alerts:",ops.alert.sum(),"of",len(ops))
print("ROC-AUC:",roc_auc_score(ops.is_anomaly,ops.anomaly_score))
print("PR-AUC:",average_precision_score(ops.is_anomaly,ops.anomaly_score))
print(classification_report(ops.is_anomaly,ops.alert,digits=3))

## 4. Evidence table and visualization

Provide the raw metrics that justify triage rather than returning only a black-box score.

In [ ]:
alerts=ops.nlargest(20,"anomaly_score")[["timestamp",*features,"anomaly_score","is_anomaly"]]
display(alerts)
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(ops.timestamp,ops.latency_ms,linewidth=.8)
marked=ops[ops.alert==1]
ax.scatter(marked.timestamp,marked.latency_ms,s=18,label="Flagged")
ax.set(title="Latency with anomaly-review candidates",xlabel="Time",ylabel="Latency (ms)")
ax.legend(); plt.show()
alerts.to_csv(ARTIFACT_DIR/"capstone_ops_alerts.csv",index=False)

## 5. Operational recommendation

An anomaly detector should feed evidence collection and human/automated triage, not automatically declare root cause.

In [ ]:
recommendation={
    "threshold":float(threshold),
    "review_fraction":review_fraction,
    "required_monitoring":[
        "score and feature distributions","alert volume and precision","reviewer dispositions",
        "incident recall","seasonality/context coverage","model/data version"
    ],
    "safety":"Use a fallback and retain raw evidence snapshots for every alert.",
}
print(json.dumps(recommendation,indent=2))

## Model/project card

Complete this before presenting the result:

| Field | Statement |
|---|---|
| Intended use | |
| Excluded use | |
| Data population and coverage | |
| Target/metric definition | |
| Evaluation split | |
| Baseline | |
| Primary result | |
| Known limitations | |
| Important subgroup behaviour | |
| Human review / abstention | |
| Monitoring | |
| Owner and review cadence | |

## Final reflection

1. Which result changed your initial belief?
2. Which assumption creates the largest residual risk?
3. What simpler alternative was competitive?
4. What evidence is still required before an operational decision?
5. What would you monitor first after release?

Re-run the notebook from a clean kernel and verify generated artifacts before considering the capstone complete.